# LeetCode 1395 — Count Number of Teams bằng Python

Link đề: https://leetcode.com/problems/count-number-of-teams/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Binary Indexed Tree, Segment Tree

Skeleton LeetCode:

```python
class Solution:
    def numTeams(self, rating: list[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đếm **bộ ba theo thứ tự chỉ số** `i < j < k` thoả điều kiện đơn điệu — từ thử hết `O(n³)` đến `O(n²)` và `O(n log n)`.
2. Nắm kỹ thuật **cố định phần tử giữa**: đếm số phần tử nhỏ hơn/lớn hơn ở hai bên rồi **nhân** (quy tắc nhân trong tổ hợp).
3. Hiểu cách nhìn **DP đếm dãy con tăng độ dài 2, 3** và tổng quát cho độ dài `L`.
4. Cài đặt **Fenwick Tree (Binary Indexed Tree)** để đếm "có bao nhiêu số đã gặp nhỏ hơn x" trong `O(log V)`, kèm **nén toạ độ**.
5. Tránh lỗi: quên dãy giảm, ghép sai cặp khi nhân, Fenwick dùng chỉ số 0, sắp xếp mảng làm mất thứ tự.

## 2. Đề bài & ví dụ

Có `n` người lính đứng thành hàng, người thứ `i` có điểm `rating[i]` (các điểm **đôi một khác nhau**). Chọn 3 người có chỉ số `i < j < k` lập một đội. Đội **hợp lệ** nếu

- `rating[i] < rating[j] < rating[k]` (tăng dần), **hoặc**
- `rating[i] > rating[j] > rating[k]` (giảm dần).

Trả về **số đội hợp lệ** (một người có thể thuộc nhiều đội).

**Ví dụ 1:** `rating = [2,5,3,4,1]` → `3`: các đội `(2,3,4)`, `(5,4,1)`, `(5,3,1)`.

**Ví dụ 2:** `rating = [2,1,3]` → `0`.

**Ví dụ 3:** `rating = [1,2,3,4]` → `4`: chọn 3 trong 4 số tăng dần, `C(4,3) = 4`.

**Ràng buộc:** `3 <= n <= 1000`, `1 <= rating[i] <= 10^5`, các giá trị đôi một khác nhau.

In [1]:
import random
import time
from bisect import bisect_left
from itertools import combinations
from math import comb
from typing import List

examples = [([2, 5, 3, 4, 1], 3), ([2, 1, 3], 0), ([1, 2, 3, 4], 4)]
for rating, want in examples:
    print(rating, "->", want)

[2, 5, 3, 4, 1] -> 3
[2, 1, 3] -> 0
[1, 2, 3, 4] -> 4


## 3. Kiến thức nền

### 3.1 `itertools.combinations` giữ nguyên thứ tự

`combinations(arr, 3)` sinh mọi bộ 3 phần tử theo **đúng thứ tự xuất hiện** trong `arr` (tức là ứng với `i < j < k`). Số bộ = `C(n, 3)`.

In [2]:
arr = [2, 5, 3, 4]
for t in combinations(arr, 3):
    print(t, "tăng" if t[0] < t[1] < t[2] else ("giảm" if t[0] > t[1] > t[2] else ""))
print("C(4,3) =", comb(4, 3))

(2, 5, 3) 
(2, 5, 4) 
(2, 3, 4) tăng
(5, 3, 4) 
C(4,3) = 4


### 3.2 So sánh dây chuyền `a < b < c`

Python cho viết `a < b < c`, nghĩa là `a < b and b < c`. Gọn và tránh lỗi quên một vế.

In [3]:
print(1 < 3 < 5, 1 < 5 < 3, 5 > 3 > 1, 5 > 1 > 3)

True False True False


### 3.3 Quy tắc nhân

Nếu có `a` cách chọn phần tử thứ nhất và **độc lập** `b` cách chọn phần tử thứ ba, thì có `a × b` cách chọn cặp. Đây là nền của cách "cố định phần tử giữa": với `j` cố định, số đội tăng có giữa là `j` bằng `(số i < j có rating nhỏ hơn) × (số k > j có rating lớn hơn)`.

In [4]:
left_smaller = [2, 3]      # các lựa chọn cho rating[i]
right_bigger = [7, 8, 9]   # các lựa chọn cho rating[k]
pairs = [(a, 6, b) for a in left_smaller for b in right_bigger]
print(len(pairs), "=", len(left_smaller), "x", len(right_bigger), "->", pairs)

6 = 2 x 3 -> [(2, 6, 7), (2, 6, 8), (2, 6, 9), (3, 6, 7), (3, 6, 8), (3, 6, 9)]


### 3.4 `sum(1 for ...)` để đếm có điều kiện

`sum(1 for x in arr if cond(x))` hoặc `sum(cond(x) for x in arr)` (vì `True == 1`).

In [5]:
arr = [2, 5, 3, 4, 1]
x = 3
print("số phần tử < 3:", sum(1 for v in arr if v < x), "| số phần tử > 3:", sum(v > x for v in arr))

số phần tử < 3: 2 | số phần tử > 3: 2


### 3.5 Fenwick Tree (Binary Indexed Tree)

Cấu trúc trên mảng `tree[1..N]` hỗ trợ hai thao tác `O(log N)`:

- `update(i, +1)`: đánh dấu đã gặp giá trị `i`;
- `query(i)`: tổng các đánh dấu ở `1..i` (= số giá trị đã gặp **≤ i**).

Bí quyết là `i & -i` — bit 1 thấp nhất của `i`. `update` đi lên `i += i & -i`, `query` đi xuống `i -= i & -i`. **Chỉ số bắt đầu từ 1** (với `i = 0`, `0 & -0 = 0` ⇒ lặp mãi).

In [6]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)

    def update(self, i, delta=1):
        while i <= self.n:
            self.tree[i] += delta
            i += i & -i

    def query(self, i):
        s = 0
        while i > 0:
            s += self.tree[i]
            i -= i & -i
        return s

for i in [1, 2, 3, 4, 6, 12]:
    print(f"i={i:2d} = {i:04b}b, i & -i = {i & -i}")
fw = Fenwick(10)
for v in [3, 7, 1, 9]:
    fw.update(v)
print("số giá trị đã gặp <= 5:", fw.query(5), "| <= 9:", fw.query(9), "| < 7:", fw.query(6))

i= 1 = 0001b, i & -i = 1
i= 2 = 0010b, i & -i = 2
i= 3 = 0011b, i & -i = 1
i= 4 = 0100b, i & -i = 4
i= 6 = 0110b, i & -i = 2
i=12 = 1100b, i & -i = 4
số giá trị đã gặp <= 5: 2 | <= 9: 4 | < 7: 2


### 3.6 Nén toạ độ

Giá trị tới `10^5` (hoặc `10^9` ở biến thể) thì thay mỗi giá trị bằng **thứ hạng** của nó trong mảng đã sắp xếp: `rank = bisect_left(sorted_vals, v) + 1`. Thứ tự giữ nguyên, nhưng Fenwick chỉ cần kích thước `n`.

In [7]:
rating = [200, 50000, 3, 99999, 7]
vals = sorted(rating)
ranks = [bisect_left(vals, v) + 1 for v in rating]
print("rating:", rating)
print("rank  :", ranks)

rating: [200, 50000, 3, 99999, 7]
rank  : [3, 4, 1, 5, 2]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** `[2,5,3,4,1]` — liệt kê `C(5,3) = 10` bộ:

| (i,j,k) | giá trị | tăng? | giảm? |
|---|---|---|---|
| 0,1,2 | 2,5,3 | | |
| 0,1,3 | 2,5,4 | | |
| 0,1,4 | 2,5,1 | | |
| 0,2,3 | 2,3,4 | ✔ | |
| 0,2,4 | 2,3,1 | | |
| 0,3,4 | 2,4,1 | | |
| 1,2,3 | 5,3,4 | | |
| 1,2,4 | 5,3,1 | | ✔ |
| 1,3,4 | 5,4,1 | | ✔ |
| 2,3,4 | 3,4,1 | | |

⇒ 3 đội. Nhận xét: mọi đội hợp lệ đều có **một phần tử ở giữa** (`j`). Nếu ta biết, với mỗi `j`, có bao nhiêu phần tử nhỏ hơn bên trái và lớn hơn bên phải, ta đếm được ngay mà không cần liệt kê.

In [8]:
rating = [2, 5, 3, 4, 1]
for i, j, k in combinations(range(len(rating)), 3):
    a, b, c = rating[i], rating[j], rating[k]
    tag = "tăng" if a < b < c else ("giảm" if a > b > c else "")
    if tag:
        print((i, j, k), (a, b, c), tag, "| phần tử giữa j =", j)

(0, 2, 3) (2, 3, 4) tăng | phần tử giữa j = 2
(1, 2, 4) (5, 3, 1) giảm | phần tử giữa j = 2
(1, 3, 4) (5, 4, 1) giảm | phần tử giữa j = 3


## 5. Cách 1 — Brute Force / baseline

### Ý tưởng
Ba vòng lặp lồng nhau `i < j < k`, kiểm tra điều kiện tăng hoặc giảm.

In [9]:
def teams_brute(rating):
    n = len(rating)
    cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if rating[i] < rating[j] < rating[k] or rating[i] > rating[j] > rating[k]:
                    cnt += 1
    return cnt

for rating, want in examples:
    print(rating, "->", teams_brute(rating), "| mong đợi", want)

[2, 5, 3, 4, 1] -> 3 | mong đợi 3
[2, 1, 3] -> 0 | mong đợi 0
[1, 2, 3, 4] -> 4 | mong đợi 4


Phiên bản một dòng với `combinations`:

In [10]:
def teams_combo(rating):
    return sum(1 for a, b, c in combinations(rating, 3) if a < b < c or a > b > c)

print([teams_combo(r) for r, _ in examples])

[3, 0, 4]


### Độ phức tạp và nút thắt

- Time `O(n³)` — chính xác `C(n,3) ≈ n³/6`. Với `n = 1000` là ~1,66·10⁸ bộ ⇒ quá chậm trong Python (cỡ 10 giây trở lên tuỳ máy, vượt xa giới hạn thời gian).
- Space `O(1)`.

Đo thử với `n = 150` để thấy tốc độ tăng theo luỹ thừa 3:

In [11]:
rng = random.Random(0)
for n in [50, 100, 150]:
    arr = rng.sample(range(1, 10**5 + 1), n)
    t = time.perf_counter()
    teams_brute(arr)
    print(f"n={n:3d}: {time.perf_counter() - t:.3f}s")

n= 50: 0.002s
n=100: 0.012s
n=150: 0.038s


Nút thắt: ta duyệt **từng cặp (i, k)** cho mỗi `j`, trong khi chỉ cần **số lượng** `i` hợp lệ và **số lượng** `k` hợp lệ.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight — cố định phần tử giữa.** Với mỗi `j`, đặt:

- `ls` = số `i < j` có `rating[i] < rating[j]` (trái, nhỏ hơn)
- `lg` = số `i < j` có `rating[i] > rating[j]` (trái, lớn hơn)
- `rs` = số `k > j` có `rating[k] < rating[j]` (phải, nhỏ hơn)
- `rg` = số `k > j` có `rating[k] > rating[j]` (phải, lớn hơn)

Thì số đội có giữa là `j`:

- tăng: chọn `i` trong `ls`, `k` trong `rg` ⇒ `ls × rg`
- giảm: chọn `i` trong `lg`, `k` trong `rs` ⇒ `lg × rs`

**Đáp án = Σⱼ (ls·rg + lg·rs).**

Vì các giá trị khác nhau: `ls + lg = j` và `rs + rg = n − 1 − j`, nên chỉ cần đếm `ls` và `rs`, hai cái còn lại suy ra.

**Quy tắc "nếu… thì…":** nếu biết `ls` và `rs` cho mỗi `j` trong `O(T)` thì cả bài là `O(n·T)`. Đếm thẳng: `T = n` ⇒ `O(n²)`. Dùng Fenwick: `T = log n` ⇒ `O(n log n)`.

## 7. Các cách tiếp theo

### Cách 2 — Cố định phần tử giữa, `O(n²)`

**Từng đoạn:**
1. Duyệt `j` từ `1` đến `n−2` (phần tử đầu/cuối không thể làm giữa).
2. Đếm `ls` bằng một vòng qua `i < j`, đếm `rs` bằng một vòng qua `k > j`.
3. Suy ra `lg = j − ls`, `rg = (n − 1 − j) − rs`.
4. Cộng `ls·rg + lg·rs`.

In [12]:
def teams_middle(rating):
    n = len(rating)
    total = 0
    for j in range(1, n - 1):
        x = rating[j]
        ls = sum(1 for i in range(j) if rating[i] < x)
        rs = sum(1 for k in range(j + 1, n) if rating[k] < x)
        lg = j - ls
        rg = (n - 1 - j) - rs
        total += ls * rg + lg * rs
    return total

for rating, want in examples:
    print(rating, "->", teams_middle(rating), "| mong đợi", want)

[2, 5, 3, 4, 1] -> 3 | mong đợi 3
[2, 1, 3] -> 0 | mong đợi 0
[1, 2, 3, 4] -> 4 | mong đợi 4


In [13]:
class SolutionMiddle:
    def numTeams(self, rating: List[int]) -> int:
        n = len(rating)
        total = 0
        for j in range(1, n - 1):
            x = rating[j]
            ls = rs = 0
            for i in range(j):
                if rating[i] < x:
                    ls += 1
            for k in range(j + 1, n):
                if rating[k] < x:
                    rs += 1
            total += ls * ((n - 1 - j) - rs) + (j - ls) * rs
        return total

print([SolutionMiddle().numTeams(r) for r, _ in examples])

[3, 0, 4]


Time `O(n²)` (≈ 10⁶ phép so sánh với `n = 1000`, chạy tốt), Space `O(1)`.

### Cách 3 — DP đếm dãy con đơn điệu theo độ dài

Một góc nhìn khác (gắn với tag Dynamic Programming):

- `inc2[j]` = số cặp `(i, j)` với `i < j`, `rating[i] < rating[j]` (dãy con tăng độ dài 2 **kết thúc tại** `j`).
- `inc3[k]` = Σ `inc2[j]` trên các `j < k` có `rating[j] < rating[k]` (nối thêm `k` vào sau mọi dãy tăng độ dài 2 kết thúc tại `j`).
- Tương tự `dec2`, `dec3` cho dãy giảm.

Đáp án = Σ `inc3[k]` + Σ `dec3[k]`. Ưu điểm: **tổng quát ngay** cho đội `L` người (Bài 4 mục 17).

In [14]:
def teams_dp(rating):
    n = len(rating)
    inc2 = [0] * n
    dec2 = [0] * n
    total = 0
    for k in range(n):
        for j in range(k):
            if rating[j] < rating[k]:
                inc2[k] += 1          # cặp (j, k) tăng
                total += inc2[j]      # nối k vào sau mọi cặp tăng kết thúc ở j
            else:
                dec2[k] += 1
                total += dec2[j]
    return total

for rating, want in examples:
    print(rating, "->", teams_dp(rating), "| mong đợi", want)

[2, 5, 3, 4, 1] -> 3 | mong đợi 3
[2, 1, 3] -> 0 | mong đợi 0
[1, 2, 3, 4] -> 4 | mong đợi 4


In [15]:
class SolutionDP:
    def numTeams(self, rating: List[int]) -> int:
        n = len(rating)
        inc2, dec2 = [0] * n, [0] * n
        total = 0
        for k in range(n):
            for j in range(k):
                if rating[j] < rating[k]:
                    inc2[k] += 1
                    total += inc2[j]
                else:
                    dec2[k] += 1
                    total += dec2[j]
        return total

print([SolutionDP().numTeams(r) for r, _ in examples])

[3, 0, 4]


Lưu ý: khi xử lý `k`, mọi `inc2[j]` với `j < k` đã được tính xong ở các vòng trước ⇒ thứ tự duyệt đúng. Time `O(n²)`, Space `O(n)`.

### Cách 4 — Fenwick Tree, `O(n log n)`

Ta cần `ls[j]` (số phần tử bên trái nhỏ hơn) và `rs[j]` (số phần tử bên phải nhỏ hơn) nhanh hơn `O(n)`.

**Từng đoạn:**
1. Nén toạ độ: `rank[j] ∈ [1..n]`.
2. Quét **trái → phải** với Fenwick rỗng: trước khi thêm `j`, `ls[j] = query(rank[j] − 1)`; rồi `update(rank[j])`.
3. Quét **phải → trái** với Fenwick mới: `rs[j] = query(rank[j] − 1)`; rồi `update(rank[j])`.
4. Ghép công thức `Σ ls·rg + lg·rs`.

In [16]:
def teams_fenwick(rating):
    n = len(rating)
    vals = sorted(rating)
    rank = [bisect_left(vals, v) + 1 for v in rating]

    ls = [0] * n
    fw = Fenwick(n)
    for j in range(n):
        ls[j] = fw.query(rank[j] - 1)
        fw.update(rank[j])

    rs = [0] * n
    fw = Fenwick(n)
    for j in range(n - 1, -1, -1):
        rs[j] = fw.query(rank[j] - 1)
        fw.update(rank[j])

    total = 0
    for j in range(n):
        lg = j - ls[j]
        rg = (n - 1 - j) - rs[j]
        total += ls[j] * rg + lg * rs[j]
    return total

for rating, want in examples:
    print(rating, "->", teams_fenwick(rating), "| mong đợi", want)

[2, 5, 3, 4, 1] -> 3 | mong đợi 3
[2, 1, 3] -> 0 | mong đợi 0
[1, 2, 3, 4] -> 4 | mong đợi 4


In [17]:
class SolutionFenwick:
    def numTeams(self, rating: List[int]) -> int:
        n = len(rating)
        order = {v: i + 1 for i, v in enumerate(sorted(rating))}   # nén toạ độ
        rank = [order[v] for v in rating]

        def count_smaller(seq):
            tree = [0] * (n + 1)
            res = []
            for r in seq:
                s, i = 0, r - 1
                while i > 0:
                    s += tree[i]
                    i -= i & -i
                res.append(s)
                i = r
                while i <= n:
                    tree[i] += 1
                    i += i & -i
            return res

        ls = count_smaller(rank)
        rs = count_smaller(rank[::-1])[::-1]
        return sum(ls[j] * ((n - 1 - j) - rs[j]) + (j - ls[j]) * rs[j] for j in range(n))

print([SolutionFenwick().numTeams(r) for r, _ in examples])

[3, 0, 4]


Time `O(n log n)`, Space `O(n)`. Với `n ≤ 1000` thì `O(n²)` đã đủ, nhưng Fenwick là cách duy nhất ở đây còn chạy tốt khi `n = 10⁵` (biến thể).

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1** `[2,5,3,4,1]` (n = 5):

| j | x | ls | lg | rs | rg | ls·rg | lg·rs | cộng dồn |
|---|---|---|---|---|---|---|---|---|
| 1 | 5 | 1 | 0 | 3 | 0 | 0 | 0 | 0 |
| 2 | 3 | 1 | 1 | 1 | 1 | 1 | 1 | 2 |
| 3 | 4 | 2 | 1 | 1 | 0 | 0 | 1 | 3 |

(`j = 0` và `j = 4` không làm giữa được.) Đáp án **3** ✔. Ở `j = 2` (x = 3): đội tăng `(2,3,4)` và đội giảm `(5,3,1)`; ở `j = 3` (x = 4): đội giảm `(5,4,1)`.

In [18]:
def trace_middle(rating):
    n = len(rating)
    total = 0
    print(f"{'j':>2} {'x':>4} {'ls':>3} {'lg':>3} {'rs':>3} {'rg':>3} {'+tăng':>6} {'+giảm':>6} {'tổng':>5}")
    for j in range(1, n - 1):
        x = rating[j]
        ls = sum(1 for i in range(j) if rating[i] < x)
        rs = sum(1 for k in range(j + 1, n) if rating[k] < x)
        lg, rg = j - ls, (n - 1 - j) - rs
        total += ls * rg + lg * rs
        print(f"{j:>2} {x:>4} {ls:>3} {lg:>3} {rs:>3} {rg:>3} {ls * rg:>6} {lg * rs:>6} {total:>5}")
    return total

print("Kết quả:", trace_middle([2, 5, 3, 4, 1]))
print()
print("Kết quả:", trace_middle([1, 2, 3, 4]))

 j    x  ls  lg  rs  rg  +tăng  +giảm  tổng
 1    5   1   0   3   0      0      0     0
 2    3   1   1   1   1      1      1     2
 3    4   2   1   1   0      0      1     3
Kết quả: 3

 j    x  ls  lg  rs  rg  +tăng  +giảm  tổng
 1    2   1   0   0   2      2      0     2
 2    3   2   0   0   1      2      0     4
Kết quả: 4


Trace Cách 4 (Fenwick) — in `ls`/`rs` từ hai lượt quét:

In [19]:
def trace_fenwick(rating):
    n = len(rating)
    vals = sorted(rating)
    rank = [bisect_left(vals, v) + 1 for v in rating]
    print("rating:", rating, "| rank:", rank)
    fw = Fenwick(n)
    ls = []
    print("Quét trái -> phải:")
    for j in range(n):
        ls.append(fw.query(rank[j] - 1))
        fw.update(rank[j])
        print(f"  j={j} x={rating[j]} rank={rank[j]}: ls={ls[-1]}, tree={fw.tree[1:]}")
    fw = Fenwick(n)
    rs = [0] * n
    print("Quét phải -> trái:")
    for j in range(n - 1, -1, -1):
        rs[j] = fw.query(rank[j] - 1)
        fw.update(rank[j])
        print(f"  j={j} x={rating[j]} rank={rank[j]}: rs={rs[j]}, tree={fw.tree[1:]}")
    print("ls =", ls, "| rs =", rs)
    return sum(ls[j] * ((n - 1 - j) - rs[j]) + (j - ls[j]) * rs[j] for j in range(n))

print("Kết quả:", trace_fenwick([2, 5, 3, 4, 1]))

rating: [2, 5, 3, 4, 1] | rank: [2, 5, 3, 4, 1]
Quét trái -> phải:
  j=0 x=2 rank=2: ls=0, tree=[0, 1, 0, 1, 0]
  j=1 x=5 rank=5: ls=1, tree=[0, 1, 0, 1, 1]
  j=2 x=3 rank=3: ls=1, tree=[0, 1, 1, 2, 1]
  j=3 x=4 rank=4: ls=2, tree=[0, 1, 1, 3, 1]
  j=4 x=1 rank=1: ls=0, tree=[1, 2, 1, 4, 1]
Quét phải -> trái:
  j=4 x=1 rank=1: rs=0, tree=[1, 1, 0, 1, 0]
  j=3 x=4 rank=4: rs=1, tree=[1, 1, 0, 2, 0]
  j=2 x=3 rank=3: rs=1, tree=[1, 1, 1, 3, 0]
  j=1 x=5 rank=5: rs=3, tree=[1, 1, 1, 3, 1]
  j=0 x=2 rank=2: rs=1, tree=[1, 2, 1, 4, 1]
ls = [0, 1, 1, 2, 0] | rs = [1, 3, 1, 1, 0]
Kết quả: 3


## 9. Code nộp LeetCode

Bản chính dùng Cách 2 (cố định phần tử giữa) — ngắn, `O(n²)`, đủ nhanh với `n ≤ 1000`.

In [20]:
from typing import List

class Solution:
    def numTeams(self, rating: List[int]) -> int:
        n = len(rating)
        total = 0
        for j in range(1, n - 1):
            x = rating[j]
            ls = sum(1 for i in range(j) if rating[i] < x)          # trái, nhỏ hơn
            rs = sum(1 for k in range(j + 1, n) if rating[k] < x)   # phải, nhỏ hơn
            lg = j - ls                                              # trái, lớn hơn
            rg = (n - 1 - j) - rs                                    # phải, lớn hơn
            total += ls * rg + lg * rs
        return total

for rating, want in examples:
    assert Solution().numTeams(rating) == want
print("Solution đúng với 3 ví dụ của đề")

Solution đúng với 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

**Phân hoạch theo phần tử giữa.** Mỗi đội hợp lệ `(i, j, k)` có **đúng một** phần tử giữa `j`. Vậy tập các đội được chia thành các nhóm rời nhau theo `j`; tổng số đội = tổng kích thước các nhóm.

**Đếm một nhóm.** Với `j` cố định, đội tăng `(i, j, k)` cần `i < j`, `rating[i] < rating[j]` và `k > j`, `rating[k] > rating[j]`. Hai điều kiện về `i` và về `k` **độc lập** nhau (điều kiện `rating[i] < rating[k]` tự động đúng nhờ bắc cầu qua `rating[j]`). Theo quy tắc nhân, có `ls · rg` đội tăng. Tương tự `lg · rs` đội giảm. Đội tăng và đội giảm không trùng nhau.

**Suy ra `lg`, `rg`.** Các giá trị khác nhau nên mỗi `i < j` hoặc nhỏ hơn hoặc lớn hơn `rating[j]` ⇒ `ls + lg = j`; tương tự bên phải.

**DP (Cách 3).** Bất biến: sau khi xử lý xong `k`, `inc2[k]` = số `j < k` có `rating[j] < rating[k]`. Mỗi đội tăng `(i, j, k)` được đếm đúng một lần — khi xét cặp `(j, k)`, nó nằm trong `inc2[j]` (vì `i` là một trong các phần tử trước `j` nhỏ hơn `rating[j]`).

**Fenwick (Cách 4).** Khi quét trái → phải, tại thời điểm xét `j`, cây chứa đúng các rank của `rating[0..j−1]`; `query(rank[j] − 1)` đếm số rank nhỏ hơn ⇒ `ls[j]`. Lượt ngược lại cho `rs[j]`.

## 11. Phân tích Time / Space

Gọi `n = len(rating)` (≤ 1000).

| Cách | Time | Space | Với n = 1000 |
|---|---|---|---|
| 1. Brute force | `O(n³)` (`C(n,3)` bộ) | `O(1)` | ~1,7·10⁸ bộ — quá chậm |
| 2. Phần tử giữa | `O(n²)` | `O(1)` | ~10⁶ phép so sánh |
| 3. DP độ dài 2/3 | `O(n²)` | `O(n)` | ~5·10⁵ cặp |
| 4. Fenwick | `O(n log n)` | `O(n)` | ~2·10⁴ thao tác |

Không có khác biệt average/worst: mọi cách đều duyệt cố định, không phụ thuộc dữ liệu.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [21]:
ALL = {
    "brute": teams_brute,
    "combo": teams_combo,
    "middle": teams_middle,
    "middle_submit": lambda r: SolutionMiddle().numTeams(r),
    "dp": teams_dp,
    "dp_submit": lambda r: SolutionDP().numTeams(r),
    "fenwick": teams_fenwick,
    "fenwick_submit": lambda r: SolutionFenwick().numTeams(r),
    "submit": lambda r: Solution().numTeams(r),
}
edge = [
    ([1, 2, 3], 1),               # n nhỏ nhất, tăng
    ([3, 2, 1], 1),               # giảm
    ([2, 1, 3], 0),
    ([1, 3, 2], 0),
    ([5, 4, 3, 2, 1], 10),        # giảm hoàn toàn: C(5,3)
    (list(range(1, 11)), 120),    # tăng hoàn toàn: C(10,3)
    ([1, 5, 2, 4, 3], 3),         # zíc zắc nhẹ
    ([100000, 1, 99999], 0),      # giá trị biên
]
for r, want in edge:
    assert teams_brute(r) == want, (r, teams_brute(r))
for name, f in ALL.items():
    for r, want in examples + edge:
        assert f(r) == want, (name, r, f(r), want)
print("Mọi cách đúng với", len(examples + edge), "test (nhãn edge kiểm chứng bằng brute force)")

Mọi cách đúng với 11 test (nhãn edge kiểm chứng bằng brute force)


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Sinh hoán vị/tập giá trị **khác nhau** bằng `rng.sample` để đúng ràng buộc đề.

In [22]:
rng = random.Random(1395)
for _ in range(800):
    n = rng.randint(3, 25)
    hi = rng.choice([n, 50, 10**5])
    r = rng.sample(range(1, hi + 1), n)
    want = teams_brute(r)
    for name, f in ALL.items():
        assert f(r) == want, (name, r)
print("800 mảng ngẫu nhiên: mọi cách khớp brute force")

800 mảng ngẫu nhiên: mọi cách khớp brute force


### 12.3 Giới hạn `n = 1000`

Bỏ qua brute force. Kiểm tra trường hợp tăng hoàn toàn (đáp án `C(1000, 3)`) và mảng ngẫu nhiên.

In [23]:
inc = list(range(1, 1001))
big = rng.sample(range(1, 10**5 + 1), 1000)
for name in ["middle", "dp", "fenwick", "submit"]:
    f = ALL[name]
    t = time.perf_counter()
    a, b = f(inc), f(big)
    print(f"{name:8s}: tăng={a} (C(1000,3)={comb(1000, 3)}), ngẫu nhiên={b}  ({time.perf_counter() - t:.3f}s)")
    assert a == comb(1000, 3)
assert len({ALL[nm](big) for nm in ["middle", "dp", "fenwick", "submit"]}) == 1

middle  : tăng=166167000 (C(1000,3)=166167000), ngẫu nhiên=55780244  (0.063s)
dp      : tăng=166167000 (C(1000,3)=166167000), ngẫu nhiên=55780244  (0.063s)
fenwick : tăng=166167000 (C(1000,3)=166167000), ngẫu nhiên=55780244  (0.004s)
submit  : tăng=166167000 (C(1000,3)=166167000), ngẫu nhiên=55780244  (0.068s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm đội **tăng**, quên đội giảm

In [24]:
def wrong_only_increasing(rating):
    n = len(rating)
    total = 0
    for j in range(1, n - 1):
        ls = sum(1 for i in range(j) if rating[i] < rating[j])
        rg = sum(1 for k in range(j + 1, n) if rating[k] > rating[j])
        total += ls * rg                          # SAI: thiếu lg * rs
    return total

for r in [[2, 5, 3, 4, 1], [3, 2, 1]]:
    print(r, "| sai:", wrong_only_increasing(r), "| đúng:", Solution().numTeams(r))

[2, 5, 3, 4, 1] | sai: 1 | đúng: 3
[3, 2, 1] | sai: 0 | đúng: 1


`[3,2,1]` chỉ có một đội giảm ⇒ bản sai trả 0. **Sửa:** cộng thêm `lg * rs`.

### Lỗi 2 — Ghép sai cặp khi nhân

Viết `ls * rs + lg * rg` (nhỏ-nhỏ, lớn-lớn). Bộ `(nhỏ hơn, x, nhỏ hơn)` là hình "chữ V ngược" — không đơn điệu!

In [25]:
def wrong_pairing(rating):
    n = len(rating)
    total = 0
    for j in range(1, n - 1):
        x = rating[j]
        ls = sum(1 for i in range(j) if rating[i] < x)
        rs = sum(1 for k in range(j + 1, n) if rating[k] < x)
        lg, rg = j - ls, (n - 1 - j) - rs
        total += ls * rs + lg * rg                # SAI: ghép nhầm
    return total

for r in [[1, 2, 3], [1, 3, 2], [2, 5, 3, 4, 1]]:
    print(r, "| sai:", wrong_pairing(r), "| đúng:", Solution().numTeams(r))

[1, 2, 3] | sai: 0 | đúng: 1
[1, 3, 2] | sai: 1 | đúng: 0
[2, 5, 3, 4, 1] | sai: 7 | đúng: 3


`[1,3,2]` không có đội nào, nhưng bản sai đếm `(1,3,2)` vì `1 < 3` và `2 < 3`. **Sửa:** tăng = trái nhỏ × phải **lớn**; giảm = trái lớn × phải **nhỏ**.

### Lỗi 3 — Fenwick dùng chỉ số 0 ⇒ lặp vô hạn

Không cộng 1 khi nén toạ độ (`rank` từ 0). `update(0)`: `0 & -0 == 0` nên `i` mãi bằng 0. Bản minh hoạ có **bộ đếm bước** để tự dừng.

In [26]:
def wrong_update_from_zero(tree, i, n, limit=50):
    steps = 0
    while i <= n:
        steps += 1
        if steps > limit:
            return f"i vẫn = {i} sau {limit} bước -> lặp vô hạn"
        tree[i] += 1
        i += i & -i
    return "xong"

vals = sorted([30, 10, 20])
rank0 = [bisect_left(vals, v) for v in [30, 10, 20]]       # SAI: rank bắt đầu từ 0
print("rank0 =", rank0)
print("update(rank 0):", wrong_update_from_zero([0] * 4, rank0[1], 3))
print("đúng: dùng rank + 1 =", [r + 1 for r in rank0], "| update(1):", wrong_update_from_zero([0] * 4, 1, 3))

rank0 = [2, 0, 1]
update(rank 0): i vẫn = 0 sau 50 bước -> lặp vô hạn
đúng: dùng rank + 1 = [3, 1, 2] | update(1): xong


**Sửa:** `rank = bisect_left(vals, v) + 1`, Fenwick kích thước `n + 1`.

### Lỗi 4 — Sắp xếp mảng trước rồi đếm

Sắp xếp "cho dễ" làm **mất thứ tự chỉ số** — đề yêu cầu `i < j < k` theo vị trí đứng trong hàng.

In [27]:
def wrong_sorted_first(rating):
    r = sorted(rating)                               # SAI: phá thứ tự
    return sum(1 for a, b, c in combinations(r, 3) if a < b < c or a > b > c)

for r in [[2, 1, 3], [2, 5, 3, 4, 1]]:
    print(r, "| sai:", wrong_sorted_first(r), "| đúng:", Solution().numTeams(r))

[2, 1, 3] | sai: 1 | đúng: 0
[2, 5, 3, 4, 1] | sai: 10 | đúng: 3


Sau khi sắp xếp, **mọi** bộ ba đều tăng ⇒ luôn ra `C(n,3)`. **Sửa:** giữ nguyên thứ tự; chỉ dùng mảng đã sắp để **nén toạ độ**.

### Lỗi 5 — Fenwick `query(rank)` thay vì `query(rank − 1)`

Muốn đếm số **nhỏ hơn hẳn** nhưng `query(rank)` đếm cả giá trị bằng — với các giá trị khác nhau, ở lượt quét chưa có chính nó nên không sai… nhưng ở **biến thể có trùng lặp** thì sai. Kiểm tra trên mảng có giá trị lặp (Bài 5):

In [28]:
def smaller_left(rating, strict=True):
    vals = sorted(set(rating))
    n = len(vals)
    fw = Fenwick(n)
    out = []
    for v in rating:
        r = bisect_left(vals, v) + 1
        out.append(fw.query(r - 1) if strict else fw.query(r))   # strict=False là bản SAI
        fw.update(r)
    return out

arr = [3, 1, 3, 2, 3]
truth = [sum(1 for i in range(j) if arr[i] < arr[j]) for j in range(len(arr))]
print("đúng (đếm tay):", truth)
print("query(r-1)    :", smaller_left(arr, True))
print("query(r) SAI  :", smaller_left(arr, False))

đúng (đếm tay): [0, 0, 1, 1, 2]
query(r-1)    : [0, 0, 1, 1, 2]
query(r) SAI  : [0, 0, 2, 1, 4]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm bộ ba i < j < k thoả quan hệ thứ tự"** ⇒ nghĩ ngay tới **cố định phần tử giữa** và quy tắc nhân.
- `n ≤ 1000` gợi ý `O(n²)` là đủ; `n ≤ 10⁵` thì cần Fenwick/Segment tree/merge sort.
- "Đếm số phần tử trước tôi nhỏ hơn tôi" = bài toán **đếm nghịch thế** ⇒ Fenwick + nén toạ độ.
- Giá trị khác nhau ⇒ dùng `ls + lg = j` để bớt một nửa phép đếm.
- Kiểm tra nhanh: mảng tăng dần độ dài `n` có đáp án `C(n,3)`; mảng giảm dần cũng vậy.

## 15. Biến thể của bài

1. **Đội `L` người** (dãy con tăng/giảm độ dài `L`): DP `cnt[len][k]` hoặc `L` cây Fenwick ⇒ `O(L·n log n)`. (Bài 4 mục 17.)
2. **Có giá trị trùng lặp**, chỉ đếm bộ tăng/giảm **nghiêm ngặt**: không còn `ls + lg = j`; phải đếm riêng `lg` (Bài 5).
3. **Đếm nghịch thế** (cặp `i < j`, `a[i] > a[j]`) — LeetCode 315 *Count of Smaller Numbers After Self*.
4. **Bộ ba "132"** (LeetCode 456) — chỉ hỏi có tồn tại không ⇒ stack đơn điệu `O(n)`.
5. **Increasing Triplet Subsequence** (LeetCode 334) — chỉ hỏi tồn tại bộ tăng ⇒ tham lam hai biến `O(n)`.
6. **Đếm tam giác/bộ ba có tổng/tích** — cùng ý "cố định một phần tử, đếm phần còn lại".

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi `(i,j,k)` | `O(n³)` | `O(1)` | `n` rất nhỏ; làm bản đối chiếu |
| 2. Phần tử giữa | `Σ ls·rg + lg·rs` | `O(n²)` | `O(1)` | mặc định khi `n ≤ ~3000` |
| 3. DP độ dài 2/3 | `inc3 += inc2[j]` | `O(n²)` | `O(n)` | khi cần tổng quát độ dài `L` |
| 4. Fenwick | đếm `ls`, `rs` bằng BIT | `O(n log n)` | `O(n)` | `n` lớn (10⁵), biến thể có trùng |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().numTeams([1, 3, 2, 4])` bằng bao nhiêu?

### Bài 2 (tính tay)
Lập bảng `ls, lg, rs, rg` cho `rating = [3, 1, 4, 2, 5]` và tính đáp án.

### Bài 3 (điền TODO)
Hoàn thiện `count_increasing_only(rating)`: chỉ đếm đội **tăng** bằng cách cố định phần tử giữa.

### Bài 4 (tự viết lại)
Viết `count_monotone(rating, L)`: số dãy con (theo thứ tự chỉ số) độ dài `L` **tăng nghiêm ngặt hoặc giảm nghiêm ngặt** (`L ≥ 2`, giá trị khác nhau). Với `L = 3` phải bằng `numTeams`.

### Bài 5 (mini challenge)
Viết `num_teams_dup(rating)` cho trường hợp **có thể trùng giá trị** (vẫn chỉ đếm bộ tăng/giảm **nghiêm ngặt**), chạy `O(n log n)` bằng Fenwick.

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_increasing_only(rating):
    n = len(rating)
    total = 0
    for j in range(1, n - 1):
        # TODO: ls = số i < j có rating[i] < rating[j]
        # TODO: rg = số k > j có rating[k] > rating[j]
        # TODO: total += ls * rg
        pass
    return total

# Bài 4 — TODO
def count_monotone(rating, L):
    # TODO: inc[len][k], dec[len][k]; inc[1][k] = 1
    # TODO: inc[len][k] = tổng inc[len-1][j] với j < k, rating[j] < rating[k]
    pass

# Bài 5 — TODO
def num_teams_dup(rating):
    # TODO: nén toạ độ trên set(rating); hai lượt Fenwick đếm ls, le (bằng), ... ; lg = j - ls - le
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Không có đội giảm nào (thử `3,2,?`: sau 2 chỉ có 4). (2) Đội tăng: `(1,3,4)`, `(1,2,4)`. (3) Đáp án 2.

**Bài 2.** (1) `j=1` (x=1): ls=0, lg=1, rs=0, rg=3 ⇒ 0. (2) `j=2` (x=4): ls=2, lg=0, rs=1, rg=1 ⇒ 2. (3) `j=3` (x=2): ls=1, lg=2, rs=0, rg=1 ⇒ 1. Tổng **3**: `(3,4,5)`, `(1,4,5)`, `(1,2,5)`.

**Bài 3.** (1) `sum(1 for i in range(j) if rating[i] < rating[j])`. (2) Tương tự bên phải với `>`. (3) Nhân và cộng dồn.

**Bài 4.** (1) Bảng `L × n`. (2) Hai vòng `k`, `j < k`, với mỗi độ dài `len` từ 2 đến `L`. (3) Đáp án `Σ inc[L][k] + Σ dec[L][k]`; với `L = 2` mỗi cặp được đếm đúng một lần (tăng hoặc giảm).

**Bài 5.** (1) Đếm thêm `le` = số phần tử bằng `x` bên trái = `query(r) − query(r − 1)`. (2) `lg = j − ls − le`; bên phải tương tự. (3) Đối chiếu với brute force trên mảng nhỏ nhiều trùng lặp.

## 19. Đáp án tham khảo

In [30]:
# Bài 1, 2
assert Solution().numTeams([1, 3, 2, 4]) == 2
r2 = [3, 1, 4, 2, 5]
rows = []
for j in range(1, 4):
    x = r2[j]
    ls = sum(1 for i in range(j) if r2[i] < x); rs = sum(1 for k in range(j + 1, 5) if r2[k] < x)
    rows.append((j, x, ls, j - ls, rs, 4 - j - rs))
print(rows)
assert rows == [(1, 1, 0, 1, 0, 3), (2, 4, 2, 0, 1, 1), (3, 2, 1, 2, 0, 1)]
assert Solution().numTeams(r2) == 3 == teams_brute(r2)
print("Bài 1, 2: đúng")

[(1, 1, 0, 1, 0, 3), (2, 4, 2, 0, 1, 1), (3, 2, 1, 2, 0, 1)]
Bài 1, 2: đúng


In [31]:
# Bài 3 — đáp án
def count_increasing_only_sol(rating):
    n = len(rating)
    total = 0
    for j in range(1, n - 1):
        ls = sum(1 for i in range(j) if rating[i] < rating[j])
        rg = sum(1 for k in range(j + 1, n) if rating[k] > rating[j])
        total += ls * rg
    return total

rng = random.Random(3)
for _ in range(300):
    r = rng.sample(range(1, 60), rng.randint(3, 15))
    want = sum(1 for a, b, c in combinations(r, 3) if a < b < c)
    assert count_increasing_only_sol(r) == want
print("Bài 3: đúng")

Bài 3: đúng


In [32]:
# Bài 4 — đáp án
def count_monotone_sol(rating, L):
    n = len(rating)
    inc = [[0] * n for _ in range(L + 1)]
    dec = [[0] * n for _ in range(L + 1)]
    for k in range(n):
        inc[1][k] = dec[1][k] = 1
        for j in range(k):
            for ln in range(2, L + 1):
                if rating[j] < rating[k]:
                    inc[ln][k] += inc[ln - 1][j]
                elif rating[j] > rating[k]:
                    dec[ln][k] += dec[ln - 1][j]
    if L == 1:
        return n
    return sum(inc[L]) + sum(dec[L])

def count_monotone_brute(rating, L):
    def mono(t):
        return all(t[i] < t[i + 1] for i in range(L - 1)) or all(t[i] > t[i + 1] for i in range(L - 1))
    return sum(1 for t in combinations(rating, L) if mono(t))

rng = random.Random(4)
for _ in range(300):
    r = rng.sample(range(1, 40), rng.randint(3, 11))
    for L in (2, 3, 4):
        assert count_monotone_sol(r, L) == count_monotone_brute(r, L)
    assert count_monotone_sol(r, 3) == Solution().numTeams(r)
print("đội 4 người trong [1,2,3,4,5]:", count_monotone_sol([1, 2, 3, 4, 5], 4))
print("Bài 4: đúng")

đội 4 người trong [1,2,3,4,5]: 5
Bài 4: đúng


In [33]:
# Bài 5 — đáp án
def num_teams_dup_sol(rating):
    n = len(rating)
    vals = sorted(set(rating))
    m = len(vals)
    rank = [bisect_left(vals, v) + 1 for v in rating]

    def scan(seq):
        fw = Fenwick(m)
        smaller, equal = [], []
        for r in seq:
            s = fw.query(r - 1)
            smaller.append(s)
            equal.append(fw.query(r) - s)
            fw.update(r)
        return smaller, equal

    ls, le = scan(rank)
    rs, re_ = scan(rank[::-1])
    rs, re_ = rs[::-1], re_[::-1]
    total = 0
    for j in range(n):
        lg = j - ls[j] - le[j]
        rg = (n - 1 - j) - rs[j] - re_[j]
        total += ls[j] * rg + lg * rs[j]
    return total

rng = random.Random(5)
for _ in range(500):
    r = [rng.randint(1, 6) for _ in range(rng.randint(3, 14))]
    assert num_teams_dup_sol(r) == teams_brute(r), r
for r, want in examples:
    assert num_teams_dup_sol(r) == want
print("[1,2,2,3] ->", num_teams_dup_sol([1, 2, 2, 3]))
print("Bài 5: đúng")

[1,2,2,3] -> 2
Bài 5: đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "cố định phần tử giữa":**

```python
total = 0
for j in range(1, n - 1):
    ls = sum(a[i] < a[j] for i in range(j))          # trái nhỏ
    rs = sum(a[k] < a[j] for k in range(j + 1, n))   # phải nhỏ
    lg, rg = j - ls, (n - 1 - j) - rs                # (giá trị khác nhau)
    total += ls * rg + lg * rs                       # tăng + giảm
```

**Template Fenwick đếm "bên trái nhỏ hơn":**

```python
rank = [bisect_left(vals, v) + 1 for v in a]   # từ 1!
for r in rank:
    ls.append(query(r - 1)); update(r)
```

**Checklist trước khi submit:**
- [ ] Đếm **cả hai** chiều: tăng `ls·rg` và giảm `lg·rs`.
- [ ] Ghép đúng: trái nhỏ ↔ phải lớn, trái lớn ↔ phải nhỏ.
- [ ] Không sắp xếp làm mất thứ tự chỉ số.
- [ ] Fenwick chỉ số bắt đầu từ 1; đếm "nhỏ hơn hẳn" dùng `query(r − 1)`.
- [ ] Kiểm tra mảng tăng dần cho `C(n,3)`.
- [ ] Đúng chữ ký: `numTeams(self, rating: List[int]) -> int`.